In [22]:
import pandas as pd

master = pd.read_csv(
    PROCESSED_DATA / "master_enso_monthly.csv",
    parse_dates=["Date"]
)

master.head()
print(features.columns.tolist())

['Date', 'nino34', 'nino3', 'nino4', 'iod', 'soi', 'future_nino34_lead1', 'future_nino34_lead2', 'future_nino34_lead3', 'future_nino34_lead4', 'future_nino34_lead5', 'future_nino34_lead6']


In [23]:
# FEATURE ENGINEERING
features = master.copy()

# Keep only rows where every climate index exists
features = features.dropna(
    subset=["nino34", "nino3", "nino4", "iod", "soi"]
).reset_index(drop=True)
features.info()
features.head()
features.tail()
features.shape


<class 'pandas.DataFrame'>
RangeIndex: 906 entries, 0 to 905
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    906 non-null    datetime64[us]
 1   nino34  906 non-null    float64       
 2   nino3   906 non-null    float64       
 3   nino4   906 non-null    float64       
 4   iod     906 non-null    float64       
 5   soi     906 non-null    float64       
dtypes: datetime64[us](1), float64(5)
memory usage: 42.6 KB


(906, 6)

In [29]:
# CREATE LAG FEATURES
climate_columns = ["nino34", "nino3", "nino4", "iod", "soi"]

for col in climate_columns:
    for lag in [1, 2, 3]:
        features[f"{col}_lag{lag}"] = features[col].shift(lag)
print(features.nino34_lag2)
print(features.nino34)

0       NaN
1       NaN
2     -0.23
3     -0.01
4      0.00
       ... 
892   -0.07
893   -0.14
894   -0.36
895   -0.47
896   -0.50
Name: nino34_lag2, Length: 897, dtype: float64
0     -0.23
1     -0.01
2      0.00
3      0.30
4      0.17
       ... 
892   -0.36
893   -0.47
894   -0.50
895   -0.70
896   -0.67
Name: nino34, Length: 897, dtype: float64


In [25]:
# CREATE ROLLING FEATURES
for col in climate_columns:
    features[f"{col}_roll3"] = (
        features[col]
        .rolling(window=3)
        .mean()
    )

In [26]:
# CREATE TARGET(multi- horizon)
MAX_LEAD = 6

for lead in range(1, MAX_LEAD + 1):

    features[f"future_nino34_lead{lead}"] = (
        features["nino34"].shift(-lead)
    )


In [27]:
#remove incomplete rows/missing values
features = features.dropna().reset_index(drop=True)

In [28]:
ROOT = Path("..")

PROCESSED_DATA = ROOT / "data" / "processed"

features.to_csv(
    PROCESSED_DATA /
    "enso_features_multihorizon.csv",
    index=False
)

print("Feature engineering complete.")

Feature engineering complete.
